# Word2vec

## Skip-gram

In [1]:
import nltk
from nltk.corpus import gutenberg
text = gutenberg.raw('shakespeare-hamlet.txt')
print(text[:500])

[The Tragedie of Hamlet by William Shakespeare 1599]


Actus Primus. Scoena Prima.

Enter Barnardo and Francisco two Centinels.

  Barnardo. Who's there?
  Fran. Nay answer me: Stand & vnfold
your selfe

   Bar. Long liue the King

   Fran. Barnardo?
  Bar. He

   Fran. You come most carefully vpon your houre

   Bar. 'Tis now strook twelue, get thee to bed Francisco

   Fran. For this releefe much thankes: 'Tis bitter cold,
And I am sicke at heart

   Barn. Haue you had quiet Guard?
  Fran. Not


In [2]:
def generate_pairs(text, window_size):
    corpus = text.split()
    pairs = []
    for c in range(len(corpus)):
        centre_word = corpus[c]
        start = max(0, c - window_size)
        end = min(len(corpus), c + window_size + 1)
        for j in range(start, end):
            if j == c:
                continue
            pairs.append((centre_word, corpus[j]))
    return pairs

pairs_count = generate_pairs(text, 3)
print(pairs_count[:10])

[('[The', 'Tragedie'), ('[The', 'of'), ('[The', 'Hamlet'), ('Tragedie', '[The'), ('Tragedie', 'of'), ('Tragedie', 'Hamlet'), ('Tragedie', 'by'), ('of', '[The'), ('of', 'Tragedie'), ('of', 'Hamlet')]


# Assinging index to vocab words and vice versa

In [3]:
def build_vocab(tokens):
    corpus = tokens.split()
    vocab = list(set(corpus))
    w2i = {}
    i2w = {}
    for i in range(len(vocab)):
        w2i[vocab[i]] = i
        i2w[i] = vocab[i]
    return w2i, i2w 

w, i = build_vocab(text)
print(len(w))

7422


# Initialize embedding vectors for my words

In [10]:
import numpy as np

def init_embeddings(vocab_size, d, seed =42):
    np.random.seed(42)
    v_c = np.random.randn(vocab_size, d) #matrix for centre words
    u_o = np.random.randn(vocab_size, d) #matrix for neigbor words
    return v_c, u_o

v_c, u_o = init_embeddings(len(w), 10)
print(v_c[7000])

#sigmoid function
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def score(center_idx, context_idx, v_c, u_o):
    c = v_c[center_idx]
    o = u_o[context_idx]
    return c @ np.transpose(o)

[ 0.32015241  1.1154616  -1.50523819  1.73960567  0.33008678  0.10972641
 -0.9515848  -0.63683978 -0.37754316 -1.4258539 ]


# sample K words for negative sampling

In [5]:
def get_negative_samples(text, w2i, K):
    corpus = text.split()
    uni_counter = {}
    for word in corpus:
        if word not in uni_counter:
            uni_counter[word] = 1
        else:
            uni_counter[word] += 1
    for key, value in uni_counter.items():
        uni_counter[key] = value ** (3/4)
    total = sum(uni_counter.values())
    for key, value in uni_counter.items():
        uni_counter[key] = value / total
    words = list(uni_counter.keys())
    weights = list(uni_counter.values())
    chosen = np.random.choice(words, p=weights, size=K)

    # Convert each chosen word into its index
    chosen_indices = []
    for w in chosen:
        index = w2i[w]
        chosen_indices.append(index)
    return chosen_indices

neg_samples = get_negative_samples(text, w, 4)
print(neg_samples)

[5377, 6316, 7279, 5421]


# Training step from negtaive sampling

In [12]:
def train_step(center_idx, context_idx, neg_indicies, v_c, u_o, lr):
    v = v_c[center_idx]
    o = u_o[context_idx]

    loss = -np.log(sigmoid(score(center_idx, context_idx, v_c, u_o)))
    for k in neg_indicies:
        loss += -np.log(sigmoid(-score(center_idx, k, v_c, u_o)))
    
    #gradients of my loss function
    #∂J/∂v_c = (σ(u_o^T v_c) - 1)·u_o + Σ_k σ(u_wk^T v_c)·u_wk
    #∂J/∂u_o = (σ(u_o^T v_c) - 1)·v_c
    #∂J/∂u_wk = σ(u_wk^T v_c)·v_c
    grad_v = (sigmoid(o @ v)-1)*o
    for k in neg_indicies:
        grad_v += sigmoid(u_o[k] @ np.transpose(v))*u_o[k]
    grad_o = (sigmoid(o @ v) -1)*v

    #update
    v_c[center_idx] = v - grad_v*lr
    u_o[context_idx] = o - grad_o*lr
    for k in neg_indicies:
        grad_u_k = sigmoid(u_o[k] @ np.transpose(v))*v
        u_o[k] = u_o[k] - lr*grad_u_k
    return v_c, u_o, loss
    

In [25]:

def train_word2vec(text, window_size, d, K, lr, epochs):
    w2i, i2w = build_vocab(text)
    pairs = generate_pairs(text, window_size)
    v_c, u_o = init_embeddings(len(w2i), d)

    for epoch in range(epochs):
        total_loss = 0
        for center_word, context_word in pairs:
            center_idx = w2i[center_word]
            context_idx = w2i[context_word]
            neg_indices = get_negative_samples(text, w2i, K)
            v_c, u_o, loss = train_step(center_idx, context_idx, neg_indices, v_c, u_o, lr)
            total_loss += loss
        print(f"Epoch {epoch}, Loss: {total_loss}")

    return v_c, u_o, w2i, i2w

v_c, u_o, w2i, i2w = train_word2vec(text[:5000], window_size=2, d=10, K=3, lr=0.05, epochs=10)

Epoch 0, Loss: 17407.316439378206
Epoch 1, Loss: 13488.48221843299
Epoch 2, Loss: 12168.071098390661
Epoch 3, Loss: 11425.452602731868
Epoch 4, Loss: 10794.696508488043
Epoch 5, Loss: 10296.667403863752
Epoch 6, Loss: 9783.643001235137
Epoch 7, Loss: 9231.185133580257
Epoch 8, Loss: 8773.65760080076
Epoch 9, Loss: 8238.492918054899


In [26]:
#cosine similarity
def cosine_sim(word1, word2, v_c, w2i):
    w1_vector = v_c[w2i[word1]]
    w2_vector = v_c[w2i[word2]]
    return (w1_vector @ w2_vector) / (np.linalg.norm(w1_vector)*np.linalg.norm(w2_vector))

print(cosine_sim('Hamlet', 'King', v_c, w2i))

0.6042187698557566


# CoOccurrence count

In [29]:
def cooccurrence_counts(text, window_size):
    corpus = text.split()
    counter = {}
    for i in range(len(corpus)):
        centre_word = corpus[i]
        start = max(0, i - window_size)
        end = min(len(corpus), i + window_size + 1)
        if centre_word not in counter:
            counter[centre_word] = {}
        for j in range(start, end):
            if j == i:
                continue
            neighbor = corpus[j]
            if neighbor not in counter[centre_word]:
                counter[centre_word][neighbor] = 1
            else:
                counter[centre_word][neighbor] += 1
    return counter

co = cooccurrence_counts(text, 3)
print(co['Hamlet'])

{'[The': 1, 'Tragedie': 1, 'of': 6, 'by': 1, 'William': 1, 'Shakespeare': 1, 'Though': 1, 'yet': 1, 'our': 2, 'deere': 1, 'Brothers': 1, 'Sun': 1, 'Queen.': 1, 'Good': 4, 'cast': 1, 'thy': 2, 'nightly': 1, "vnforc'd": 1, 'accord': 1, 'Sits': 1, 'smiling': 1, 'to': 4, 'touching': 1, 'the': 5, 'L[ord].': 1, 'Polon.': 1, 'Marry,': 1, 'well': 1, 'in': 3, 'this.': 1, 'Now': 1, 'heare:': 1, "It's": 1, 'giuen': 1, 'Marcellus.': 1, 'Mar.': 1, 'Lord': 5, 'Hor.': 1, 'Heauen': 1, 'secure': 1, 'a': 5, 'man': 1, 'as': 1, 'is,': 1, 'May': 1, 'doe': 1, 'my': 3, 'Chamber,': 2, 'with': 2, 'his': 2, 'doublet': 1, 'Gentlemen': 1, 'where': 1, 'is': 5, 'Guil.': 1, 'Heauens': 1, 'Came': 1, 'this': 1, 'from': 2, 'her': 2, 'Pol.': 1, 'did': 1, 'bespeake': 1, 'Prince': 1, 'try': 1, 'it.': 1, 'Enter': 4, 'reading': 1, 'on': 1, 'closely': 1, 'sent': 1, 'for': 1, 'hither,': 1, 'That': 1, 'he,': 1, 'vs,': 1, 'what': 2, 'saide,': 1, 'We': 1, 'heard': 1, 'Lights.': 1, 'Exeunt.': 2, 'Manet': 1, '&': 1, 'Horatio.': 1,